# RNN for Text Classification

## 1. Import Libraries

In [1]:
import torch
import torch.nn as nn
import numpy as np

## 2. Prepare Dataset

In [2]:
texts = [
    "i love this movie",
    "this film is great",
    "i really like this",
    "this is amazing",
    "i hate this movie",
    "this film is bad",
    "i do not like this",
    "this is terrible"
]

labels = [1, 1, 1, 1, 0, 0, 0, 0]

## 3. Preprocess Text Data

In [3]:
# Tokenize texts
tokenized = [text.split() for text in texts]

# Build vocabulary
vocab = {'<PAD>': 0}
for tokens in tokenized:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

# Convert texts to sequences
sequences = []
for tokens in tokenized:
    seq = [vocab[token] for token in tokens]
    sequences.append(seq)

# Pad sequences
max_len = max(len(seq) for seq in sequences)
padded = np.array([seq + [0] * (max_len - len(seq)) for seq in sequences])

# Convert to tensors
X = torch.tensor(padded, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)

## 4. Define RNN Model

In [4]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=16):
        super(RNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        emb = self.embedding(x)
        _, hidden = self.rnn(emb)
        out = self.fc(hidden.squeeze(0))
        out = self.sigmoid(out)
        return out

model = RNNClassifier(vocab_size=len(vocab))

## 5. Train the Model

In [5]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    
    print(f'Epoch {epoch + 1}, Loss: {loss.item():.4f}')

Epoch 1, Loss: 0.6859
Epoch 2, Loss: 0.6300
Epoch 3, Loss: 0.5805
Epoch 4, Loss: 0.5332
Epoch 5, Loss: 0.4849
Epoch 6, Loss: 0.4336
Epoch 7, Loss: 0.3781
Epoch 8, Loss: 0.3194
Epoch 9, Loss: 0.2632
Epoch 10, Loss: 0.2174


## 6. Make Predictions

In [6]:
model.eval()
with torch.no_grad():
    predictions = model(X)
    predicted_labels = (predictions > 0.5).int().squeeze().tolist()

print(f'Predictions: {predicted_labels}')

Predictions: [1, 1, 1, 1, 0, 0, 0, 0]
